1) **Set up**
- Selenium installation
- Import required libraries

In [8]:
!pip install selenium


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [68]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import time
from selenium.common.exceptions import ElementClickInterceptedException, NoSuchElementException
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd
from urllib.parse import urlparse

2. Definition of a function to do **automated scroll down** of the web page

In [69]:
def scroll_click(driver):
    """Scroll down to bottom of the page and click 'CARICA PIÙ PRODOTTI' if available"""
    last_height = driver.execute_script("return document.body.scrollHeight")

    while True:
        try:
            load_more_button = driver.find_element("id", "nextpage")
            if load_more_button.is_displayed():
                load_more_button.click()
                time.sleep(3)  
        except (NoSuchElementException, ElementClickInterceptedException):
            pass

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

In [70]:
def close_newsletter_banner(driver):
    try:
        close_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CLASS_NAME, 'btn-close'))
        )
        close_button.click()
        print("Banner close button clicked")
    except Exception as e:
        print(f"Error while trying to close the banner: {e}")

3. Initializing the **WebDriver** and sending it to the desired page

In [73]:
driver = driver = webdriver.Chrome()
Expert_home = 'https://www.expert.it/it/it/exp/?r=1'
driver.get(Expert_home)

In [74]:
# wait for the banner to show (it takes almost 5 sec)
time.sleep(6)
close_newsletter_banner(driver)

Banner close button clicked


4) Collecting the **URLs** that lead to the pages of the subcategories

In [75]:
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

In [76]:
subcategories_tags = soup.find_all('div', class_ = 'domino-menu-item')

subcategories=[]
subcategories_links = []

for i in subcategories_tags:
    link = i.find('a').get('href')
    subcategories_links.append(link)
    subcategory = i.find('a').get_text()
    subcategories.append(subcategory)
    
subcategories_links = ['https://www.expert.it' + item for item in subcategories_links if 'www.expert' not in item]

In [77]:
subcategories

['Smartphone & Cellulari',
 'Wearable',
 'Accessori Smartphone & Cellulari',
 'Telefoni Domestici',
 'Tablet',
 'E-Reader',
 'Computer Portatili',
 'PC Desktop & Monitor',
 'Stampanti e Multifunzione',
 'Networking',
 'Hard Disk & Storage',
 'Accessori Informatica',
 'Software e Antivirus',
 'TV',
 'Audio Home System',
 'Dvd & Blu Ray',
 'Decoder',
 'Hi Fi',
 'Audio Mobile',
 'Fotografia',
 'Videocamere',
 'Libera Installazione',
 'Incasso',
 'Riscaldamento',
 'Condizionatori',
 'Trattamento Aria',
 'Preparazione Cibi',
 'Cottura Cibi',
 'Caffè & Colazione',
 'Pulizia della Casa',
 'Acqua & Bevande',
 'Salute & Benessere',
 'Cura della Persona',
 'Stiro & Cucito',
 'Playstation',
 'Xbox',
 'Nintendo',
 'Altre console e game',
 'Accessori Informatica Gaming',
 'Giochi',
 'Mobilità',
 'Tempo Libero',
 'Navigazione GPS',
 'Smart Home',
 'Casa',
 'Bricolage & Materiale Elettrico']

5) Function that collects **products** from the pages

In [78]:
def find_products(soup):
    prods=soup.find_all('h5', class_='productName ms-3')

    
    products = []
    
    for tag in prods:
        product=tag.find('a').get_text(strip=True)
        products.append(product)
    
    return products

6. Function that collects the **prices** of products

In [79]:
def find_prices(soup):
    all_prices = soup.find_all('div', class_='productList-purchase-prices')
    prices = []

    for tag in all_prices:
        full_price_row = tag.find('div', class_='full-price-row')
        if full_price_row:
            integer = full_price_row.find('span', class_='full-price')
            decimal = full_price_row.find('span', class_='full-price-lower')
        else:
            integer = tag.find('span', class_='sell-price')
            decimal = tag.find('span', class_='sell-price-lower')

        if integer:
            price = integer.get_text(strip=True)
            if decimal:
                price += decimal.get_text(strip=True)
            prices.append(price)

    return prices

7. Iterative collection of each **product** and **price** (in lists) from each page of each **subcategory**

In [80]:
all_products = []
all_prices = []
categories_column = []

In [81]:
for i in range(len(subcategories_links)):
    link = subcategories_links[i]
    subcategory = subcategories[i]
    driver.get(link)
    scroll_click(driver)
    scroll_click(driver)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    cat_products = find_products(soup)
    cat_prices = find_prices(soup)
    cat = [subcategory]*len(cat_products)

    print(subcategory, len(cat_products), len(cat_prices), len(cat))
    
    for i in cat_products:
        all_products.append(i)
        
    for i in cat:
        categories_column.append(i)
        
    for i in cat_prices:
        all_prices.append(i)

Smartphone & Cellulari 491 491 491
Wearable 189 189 189
Accessori Smartphone & Cellulari 1008 1008 1008
Telefoni Domestici 107 107 107
Tablet 136 136 136
E-Reader 9 9 9
Computer Portatili 170 170 170
PC Desktop & Monitor 77 77 77
Stampanti e Multifunzione 368 368 368
Networking 187 187 187
Hard Disk & Storage 77 77 77
Accessori Informatica 723 723 723
Software e Antivirus 18 18 18
TV 835 835 835
Audio Home System 469 469 469
Dvd & Blu Ray 32 32 32
Decoder 54 54 54
Hi Fi 272 272 272
Audio Mobile 36 36 36
Fotografia 104 104 104
Videocamere 32 32 32
Libera Installazione 216 216 216
Incasso 596 596 596
Riscaldamento 135 135 135
Condizionatori 125 125 125
Trattamento Aria 170 170 170
Preparazione Cibi 456 456 456
Cottura Cibi 366 366 366
Caffè & Colazione 216 216 216
Pulizia della Casa 352 352 352
Acqua & Bevande 36 36 36
Salute & Benessere 142 142 142
Cura della Persona 445 445 445
Stiro & Cucito 153 153 153
Playstation 216 216 216
Xbox 104 104 104
Nintendo 153 153 153
Altre console e game

8. Saving the lists as columns of a **new dataset**

In [82]:
ExpertDataset = pd.DataFrame()
ExpertDataset['Product'] = all_products
ExpertDataset['Unit Price'] = all_prices
ExpertDataset['Category'] = categories_column

In [83]:
ExpertDataset

,Product,Unit Price,Category
0,APPLE IPHONE 16 128GB NERO,"888,90€",Smartphone & Cellulari
1,SAMSUNG GALAXY S24 8+256GB ONYX BLACK,"707,90€",Smartphone & Cellulari
2,SAMSUNG GALAXY S25 ULTRA 12+512GB TITANIUM SIL...,"1.614,90€",Smartphone & Cellulari
3,XIAOMI REDMI NOTE 14 5G MIDNIGHT BLACK 8G RAM ...,"255,90€",Smartphone & Cellulari
4,"MOTOROLA EDGE 50 NEO, 8/256GB, FOTOCAMERA SONY...","341,90€",Smartphone & Cellulari
...,...,...,...
10541,VARTA 6206301401,"11,99€",Bricolage & Materiale Elettrico
10542,VARTA 6016101401,"4,50€",Bricolage & Materiale Elettrico
10543,VARTA 6430101401,"2,49€",Bricolage & Materiale Elettrico
10544,VARTA PILA BOTTONE X ELETRONICA CR2450 LITIO,"3,49€",Bricolage & Materiale Elettrico


In [84]:
ExpertDataset.to_csv('/Users/camillagentili/Downloads/DataManagement/ExpertScarped.csv')